# 01 - Extração do SQL Server para CSV no MinIO

Extraímos todas as cinco tabelas do banco LojaDB.
Cada tabela gera um arquivo CSV com cabeçalho no bucket landing-zone.

Os dados continuam disponíveis no SQL Server; esta etapa grava uma
cópia dos registros no armazenamento de objetos do MinIO.

## 1. Configuração e conexões

Usamos pymssql para acessar o banco e boto3 para acessar o MinIO
pela API compatível com S3.

In [1]:
import os
import io
import csv
from pathlib import Path
import pymssql
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv

raiz = Path.cwd()
if not (raiz / "pyproject.toml").is_file():
    raiz = raiz.parent

load_dotenv(raiz / ".env", override=True)

conexao = pymssql.connect(
    server=os.environ["SQLSERVER_HOST"],
    port=os.environ["SQLSERVER_PORT"],
    user=os.environ["SQLSERVER_USER"],
    password=os.environ["SQLSERVER_PASSWORD"],
    database=os.environ["SQLSERVER_DATABASE"],
    charset="UTF-8",
    login_timeout=15,
    timeout=60
)
cursor = conexao.cursor()

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name="us-east-1",
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"},
        connect_timeout=10,
        read_timeout=30
    )
)

bucket = "landing-zone"
print("Conexões com SQL Server e MinIO preparadas.")

Conexões com SQL Server e MinIO preparadas.


## 2. Descobrir as tabelas e preparar o bucket

Consultamos INFORMATION_SCHEMA para descobrir as tabelas existentes.
Todas as tabelas do nosso banco estão no schema dbo.

O bucket landing-zone recebe os arquivos extraídos da origem.
Se já existir, ele será reutilizado.

In [2]:
cursor.execute("""
    SELECT TABLE_SCHEMA, TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_SCHEMA, TABLE_NAME
""")
tabelas = cursor.fetchall()

assert len(tabelas) >= 5, "O banco precisa conter pelo menos cinco tabelas."

for schema, tabela in tabelas:
    print(f"Tabela encontrada: {schema}.{tabela}")

try:
    s3.head_bucket(Bucket=bucket)
except ClientError as erro:
    status = erro.response["ResponseMetadata"]["HTTPStatusCode"]
    if status == 404:
        s3.create_bucket(Bucket=bucket)
    else:
        raise

print(f"Bucket preparado: {bucket}")

Tabela encontrada: dbo.categorias
Tabela encontrada: dbo.clientes
Tabela encontrada: dbo.itens_pedido
Tabela encontrada: dbo.pedidos
Tabela encontrada: dbo.produtos
Bucket preparado: landing-zone


## 3. Extrair os registros e gravar os CSVs

Para cada tabela, SELECT recupera os registros. O módulo csv escreve
os nomes das colunas e os valores em formato CSV.

put_object envia o arquivo ao MinIO. Cada tabela do schema dbo gera
um arquivo com seu nome, como produtos.csv.

Reexecutar esta etapa substitui o conteúdo dos mesmos arquivos pela
extração atual, evitando criar cópias duplicadas.

In [3]:
resumo = []

for schema, tabela in tabelas:
    schema_sql = schema.replace("]", "]]")
    tabela_sql = tabela.replace("]", "]]")
    cursor.execute(f"SELECT * FROM [{schema_sql}].[{tabela_sql}]")

    colunas = [coluna[0] for coluna in cursor.description]
    registros = cursor.fetchall()

    buffer = io.StringIO()
    escritor = csv.writer(buffer, lineterminator="\n")
    escritor.writerow(colunas)
    escritor.writerows(registros)

    conteudo = buffer.getvalue().encode("utf-8")
    arquivo = (
        f"{tabela}.csv" if schema == "dbo"
        else f"{schema}.{tabela}.csv"
    )

    s3.put_object(
        Bucket=bucket,
        Key=arquivo,
        Body=conteudo,
        ContentType="text/csv; charset=utf-8"
    )

    resumo.append((arquivo, len(registros), colunas))
    print(f"{arquivo}: {len(registros)} registros enviados.")

cursor.close()
conexao.close()

categorias.csv: 2 registros enviados.
clientes.csv: 5 registros enviados.
itens_pedido.csv: 10 registros enviados.
pedidos.csv: 5 registros enviados.
produtos.csv: 10 registros enviados.


## 4. Validar a extração

Lemos os CSVs diretamente do MinIO e conferimos o cabeçalho e a
quantidade de registros de cada tabela. Assim verificamos o conteúdo
gravado no bucket, além da execução do envio.

In [4]:
total = 0

for arquivo, quantidade, colunas in resumo:
    objeto = s3.get_object(Bucket=bucket, Key=arquivo)
    texto = objeto["Body"].read().decode("utf-8")
    linhas = list(csv.reader(io.StringIO(texto)))

    assert linhas[0] == colunas, f"Cabeçalho diferente: {arquivo}"
    assert len(linhas) - 1 == quantidade, f"Contagem diferente: {arquivo}"

    total += quantidade
    print(f"Validado: {arquivo} — {quantidade} registros")

print(f"Extração concluída: {len(resumo)} arquivos, {total} registros.")
print("Destino: bucket landing-zone no MinIO.")

Validado: categorias.csv — 2 registros
Validado: clientes.csv — 5 registros
Validado: itens_pedido.csv — 10 registros
Validado: pedidos.csv — 5 registros
Validado: produtos.csv — 10 registros
Extração concluída: 5 arquivos, 32 registros.
Destino: bucket landing-zone no MinIO.
